# Ibà on N-ATLAS: live demo server

Serves **N-ATLaS** with the **Ibà-ATLAS** adapter (`natlas/serve.py`, an OpenAI-compatible server on Transformers + PEFT), runs the Ibà app against it, and opens a temporary public link for the demo.

**Kaggle settings:** GPU **T4 x2**, Internet **On**. Secrets ticked for this notebook: `HF_TOKEN`, `NEBIUS_API_KEY` (guideline-search embeddings only; all text generation is N-ATLaS), `TAVILY_API_KEY` (optional), `HF_INDEX_REPO`.

Run the cells in order. The link dies when the session stops.

## 0. Setup (≈2 min)

In [ ]:
import os, sys, subprocess, time, re, json
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
for k in ("HF_TOKEN", "NEBIUS_API_KEY", "TAVILY_API_KEY", "HF_INDEX_REPO"):
    try:
        os.environ[k] = sec.get_secret(k); print(k, "set")
    except Exception:
        print(k, "MISSING")
!pip install -q -U "transformers>=4.46" peft accelerate fastapi uvicorn openai uv lm-format-enforcer
# Kaggle ships torchao 0.10; recent peft refuses to load next to it, and we do not use it.
!pip uninstall -y -q torchao
!rm -rf /kaggle/working/iba-natlas && git clone -q https://github.com/jemiridaniel/iba-natlas.git /kaggle/working/iba-natlas
!git -C /kaggle/working/iba-natlas log -1 --oneline

## 1. Start the model server (≈3–6 min: 16 GB download, then load across both T4s)

The server constrains each reply to the app's JSON schema (`natlas/constrain.py`); the first start also maps the 128K-token vocabulary for that (≈1 min).

In [ ]:
REPO = "/kaggle/working/iba-natlas"
server = subprocess.Popen([sys.executable, "-m", "natlas.serve", "--port", "8000"], cwd=REPO,
                          stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)
import urllib.request
for i in range(60):
    if server.poll() is not None:
        print("server exited:"); break
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print(f"model server ready after {i * 10} s"); break
    except Exception:
        time.sleep(10)
!tail -n 8 /kaggle/working/server.log
!nvidia-smi --query-gpu=index,memory.used --format=csv

## 2. Smoke test: Ibà-ATLAS vs base N-ATLaS on one case

In [ ]:
from openai import OpenAI
sys.path.insert(0, REPO)
from natlas.prompts import SYSTEM_PROMPT
llm = OpenAI(base_url="http://localhost:8000/v1", api_key="x")
case = "Pikin 3 years, hot body 4 days, e no fit drink, e dey vomit everything"
for model in ("iba-atlas", "NCAIR1/N-ATLaS"):
    t = time.time()
    r = llm.chat.completions.create(model=model, temperature=0, max_tokens=200, messages=[
        {"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": case}])
    print(f"--- {model} ({time.time() - t:.1f}s)\n{r.choices[0].message.content}\n")

## 3. Start the Ibà app on N-ATLAS (≈2–4 min)

In [ ]:
APP = f"{REPO}/app"
env = """APP_ENV=prod
CHAT_BASE_URL=http://localhost:8000/v1
LLM_STRUCTURED_OUTPUT=json_schema
MODEL_FAST=NCAIR1/N-ATLaS
MODEL_MID=NCAIR1/N-ATLaS
MODEL_REASON=NCAIR1/N-ATLaS
MODEL_ATLAS=iba-atlas
REASONING_REASON=false
MAX_TOKENS_REASONING_OFF=2048
MAX_TOKENS_FALLBACK=2048
LLM_TIMEOUT_S=600
LLM_MAX_RETRIES=0
PATIENT_EXPLANATION=true
MODEL_EMBED=Qwen/Qwen3-Embedding-8B
MAX_SPEND_USD=1.0
MODEL_PRICES={"NCAIR1/N-ATLaS":{"input":0,"output":0},"iba-atlas":{"input":0,"output":0},"Qwen/Qwen3-Embedding-8B":{"input":0.01,"output":0}}
"""
open(f"{APP}/.env", "w").write(env)
p = subprocess.run("uv sync -q && uv run -q python -m backend.app.rag.fetch_remote_index", shell=True, cwd=APP,
                   capture_output=True, text=True)
print(p.stdout[-800:], p.stderr[-800:])
app = subprocess.Popen(["uv", "run", "-q", "uvicorn", "backend.app.main:app", "--host", "0.0.0.0", "--port", "8080"],
                       cwd=APP, stdout=open("/kaggle/working/iba.log", "w"), stderr=subprocess.STDOUT,
                       start_new_session=True)
for i in range(30):
    try:
        urllib.request.urlopen("http://localhost:8080/health", timeout=5); print("app ready"); break
    except Exception:
        time.sleep(2)
!tail -n 5 /kaggle/working/iba.log

## 4. One full case through the pipeline (≈4–8 min on T4s; the reasoning step is the slow one)

In [ ]:
import requests
body = {"text": "Adult woman 28 years, 62 kg, fever 2 days, RDT positive, eating and drinking well",
        "state": "Lagos", "patient_language": "yo"}
t = time.time()
r = requests.post("http://localhost:8080/triage", json=body, timeout=1800).json()
print(f"{r['triage_label']} ({time.time() - t:.0f}s) | {r['triage_rationale']}")
print("Ibà-ATLAS:", r.get("atlas"))
print("Yoruba:", (r.get("patient_explanation") or {}).get("text"))
for s in r["decision_trace"]["steps"]:
    print(f"{s['step']:14} {str(s['model']):16} {s['latency_ms']/1000:6.1f}s  {s.get('note') or ''}"[:160])
print("\n--- model server log ---")
!tail -n 12 /kaggle/working/server.log

In [ ]:
body = {"text": "Ọmọ ọdún mẹ́ta, ara rẹ̀ gbóná fún ọjọ́ mẹ́rin, kò lè mu omi, ó ń bì gbogbo nǹkan tó bá jẹ",
        "state": "Oyo", "patient_language": "yo", "skip_questions": True}
t = time.time()
r = requests.post("http://localhost:8080/triage", json=body, timeout=1800).json()
print(r["status"], f"({time.time() - t:.0f}s) |", r["triage_label"], "|", r["triage_rationale"])
print("Danger signs:", [(h["code"], h["source"]) for h in r["danger_signs"]])
print("Ibà-ATLAS:", r.get("atlas"))
print("Yoruba:", (r.get("patient_explanation") or {}).get("text"))

## 5. Public link for the demo video (temporary)

In [ ]:
   !git -C /kaggle/working/iba-natlas pull -q && git -C /kaggle/working/iba-natlas log -1 --oneline
   app.terminate(); app.wait()
   app = subprocess.Popen(["uv", "run", "-q", "uvicorn", "backend.app.main:app", "--host", "0.0.0.0", "--port", "8080"],
                          cwd=APP, stdout=open("/kaggle/working/iba.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
   time.sleep(8); print(urllib.request.urlopen("http://localhost:8080/health").read())

In [ ]:
!wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /kaggle/working/cloudflared
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:8080"],
                          stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(12)
print(re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read()))